# multi agent를 이용한 고객 응대 chatbot

## 환경 설정

In [1]:
# 필요한 모듈 / 클래스 / 함수 임포트
import os
from typing import Annotated
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END, add_messages

C:\Users\playdata2\AppData\Local\Temp\ipykernel_17816\3377707963.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [2]:
# .env file -> API Key 가져오기
file_path='.env'
output=load_dotenv(file_path, override=True)
print(output)

True


In [3]:
# 2개의 chain 구성 -> 공통 LLM 생성
llm = ChatOpenAI(model="gpt-5-nano")

## RAG 시스템 기반 설정

### 데이터 로드 및 변환

In [4]:
# 파일 경로 설정
csv_path = "faq_manual.csv"

# CSV 파일 로드 및 변환 도구 생성 -> CSVLoader 클래스 호출
loader = CSVLoader(
    file_path=csv_path,
    encoding="utf-8",
    # 본문(page_content)에 포함할 컬럼 지정
    content_columns=["question", "answer"]
)

# load() 함수 호출 -> Document로 변환 (행별로 Document 생성)
docs = loader.load()

print(f"로드된 총 문서 수: {len(docs)}개")
print("-" * 60)
print(f"[첫 번째 Document 본문(page_content)]\n{docs[0].page_content}")
print("-" * 60)
print(f"[첫 번째 Document 메타데이터(metadata)]\n{docs[0].metadata}")

로드된 총 문서 수: 10개
------------------------------------------------------------
[첫 번째 Document 본문(page_content)]
question: 코드크래프터스 아카데미에 어떻게 가입하나요?
answer: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.
------------------------------------------------------------
[첫 번째 Document 메타데이터(metadata)]
{'source': 'faq_manual.csv', 'row': 0}


### 텍스트 임베딩 모델 생성

In [5]:
# 모델 이름 설정
model_name = "text-embedding-3-small"

# OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model=model_name
)

### 백터 저장소 및 검색 도구 생성 

In [6]:
# 백터 저장소 저장 경로 설정
chroma_path = 'chroma_db_multi'

# 이미 생성된 백터 저장소가 있다면 불러오고, 없다면 새로 생성
if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(
        persist_directory=chroma_path,
        embedding_function=model
    )
    print("기존 벡터 저장소를 성공적으로 불러왔습니다.")
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=model,
        persist_directory=chroma_path
    )
    print("신규 벡터 저장소를 생성하고 저장했습니다.")

기존 벡터 저장소를 성공적으로 불러왔습니다.


In [7]:
# 검색 도구 생성 -> as_retriever() 함수 호출
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})

In [8]:
# 검색 도구 성능 테스트
user_query = "환불은"
output = retriever.invoke(user_query)
print(f"검색 결과: \n{output}")

검색 결과: 
[Document(id='67981998-ad66-438b-a095-d948db7c9fe8', metadata={'row': 1, 'source': 'faq_manual.csv'}, page_content='question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?\nanswer: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.')]


## 에이전트별 전용 체인 생성

In [12]:
# [질문 재구성 체인] 모호한 질문을 명확한 '질문'으로 변환

## system 프롬프트
rephrase_system_prompt = """
주어진 채팅 기록(chat history)과 최신 사용자 질문이 있을 때,
이 질문이 채팅 기록의 맥락을 참조하고 있다면 모호한 질문(예: "그거 전액 다 받으려면 조건이 뭐야?"),
채팅 기록 없이도 이해할 수 있는 명확한 질문(예: 수강 신청한 강의를 전액 환불 받으려면 조건이 뭐야?)으로 다시 작성하세요.
질문에 답하지 말고, 오직 질문을 재구성하거나 그대로 반환하기만 하세요.
"""

## 전체 프롬프트 생성
rephrase_prompt = ChatPromptTemplate.from_messages(
    messages=[
        ("system", rephrase_system_prompt),
        # invoke 함수 실행 전 -> 대화 기록을 리스트 자료형(변수 이름=variable_name)으로 누적
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

print(rephrase_prompt)

print('-'*80)

## 질문 재구성 chain 생성
rephrase_chain = rephrase_prompt | llm | StrOutputParser()

input_variables=['chat_history', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messag